# Virtual Multi-View Thermal Tracking Simulation

This notebook runs the full simulation pipeline: eagle motion → thermal rendering → voxel fusion → tracking → evaluation.

**To run on Google Colab:** Click `Runtime > Run all` or run cells one by one with `Shift+Enter`.

## 1. Install Dependencies

In [ ]:
!pip install -q git+https://github.com/AliTranscrypts/wave.git@claude/start-app-implementation-oOMaO

## 2. Download Default Config

In [ ]:
import yaml

default_config = {
    "world": {
        "origin": [0.0, 0.0, 0.0],
        "x_min": -2500.0, "x_max": 2500.0,
        "y_min": -2500.0, "y_max": 2500.0,
        "z_min": 0.0, "z_max": 500.0
    },
    "eagle": {
        "temperature": 35.0,
        "radius": 0.5,
        "max_speed": 20.0,
        "max_acceleration": 5.0,
        "motion_type": "lissajous",
        "altitude_range": [50.0, 200.0],
        "lissajous_amplitudes": [300.0, 300.0, 30.0],
        "lissajous_frequencies": [0.05, 0.07, 0.03],
        "lissajous_phases": [0.0, 1.57, 0.0],
        "lissajous_z_center": 100.0,
        "initial_position": [0.0, 0.0, 100.0]
    },
    "cameras": [
        {"id": "cam_00", "position": [500.0, 0.0, 10.0], "orientation_mode": "look_at",
         "look_at_target": [0.0, 0.0, 100.0], "intrinsics_preset": "flir_640x512", "hfov_deg": 45.0},
        {"id": "cam_01", "position": [-250.0, 433.0, 10.0], "orientation_mode": "look_at",
         "look_at_target": [0.0, 0.0, 100.0], "intrinsics_preset": "flir_640x512", "hfov_deg": 45.0},
        {"id": "cam_02", "position": [-250.0, -433.0, 10.0], "orientation_mode": "look_at",
         "look_at_target": [0.0, 0.0, 100.0], "intrinsics_preset": "flir_640x512", "hfov_deg": 45.0}
    ],
    "rendering": {
        "sky_temperature": -10.0,
        "ground_temperature": 15.0,
        "atmospheric_attenuation_coeff": 0.0005,
        "vignetting_strength": 0.0,
        "blur_sigma_pixels": 2.0,
        "rendering_method": "projection",
        "noise": {"enabled": False, "gaussian_std": 0.05, "shot_noise_enabled": False,
                  "fixed_pattern_noise_std": 0.0, "random_seed": 42}
    },
    "run": {"num_frames": 100, "dt": 0.1, "random_seed": 42}
}

import os
os.makedirs("configs", exist_ok=True)
with open("configs/default_config.yaml", "w") as f:
    yaml.dump(default_config, f, default_flow_style=False)

print("Config saved.")

## 3. Run the Simulation Engine

In [ ]:
from thermal_tracker.config import load_config
from thermal_tracker.engine import SimulationEngine, SimulationOutputMode

config = load_config("configs/default_config.yaml")
config.run.num_frames = 100

engine = SimulationEngine(config)
result = engine.run(mode=SimulationOutputMode.BATCH)

print(f"Generated {result.num_frames} frames in {result.wall_clock_seconds:.2f}s")
print(f"Cameras: {[c.id for c in engine.cameras]}")
print(f"Trajectory shape: {result.trajectory.shape}")

# Show trajectory bounds so we can pick a good ROI
traj = result.trajectory
print(f"\nEagle position range:")
print(f"  X: [{traj[:,1].min():.0f}, {traj[:,1].max():.0f}] m")
print(f"  Y: [{traj[:,2].min():.0f}, {traj[:,2].max():.0f}] m")
print(f"  Z: [{traj[:,3].min():.0f}, {traj[:,3].max():.0f}] m")

## 4. Visualize the Eagle Trajectory (3D)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

traj = result.trajectory
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Eagle trajectory
ax.plot(traj[:, 1], traj[:, 2], traj[:, 3], 'b-', linewidth=1.5, label='Eagle path')
ax.scatter(traj[0, 1], traj[0, 2], traj[0, 3], c='green', s=80, marker='o', label='Start')
ax.scatter(traj[-1, 1], traj[-1, 2], traj[-1, 3], c='red', s=80, marker='x', label='End')

# Camera positions
for cam_cfg in config.cameras:
    pos = cam_cfg.position
    ax.scatter(pos[0], pos[1], pos[2], c='orange', s=100, marker='^')
    ax.text(pos[0], pos[1], pos[2] + 10, cam_cfg.id, fontsize=8)

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.set_title('Eagle Trajectory & Camera Positions')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Visualize Thermal Images

In [ ]:
# Show thermal images from all 3 cameras at frame 50
frame = result.frame_bundles[50]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, (cam_id, img) in enumerate(frame.camera_images.items()):
    im = axes[i].imshow(img, cmap='inferno', aspect='auto')
    axes[i].set_title(f"{cam_id} (frame 50)")
    # Mark ground truth projection
    if cam_id in frame.ground_truth_2d and frame.visibility.get(cam_id, False):
        u, v = frame.ground_truth_2d[cam_id]
        axes[i].plot(u, v, 'g+', markersize=15, markeredgewidth=2)
    plt.colorbar(im, ax=axes[i], label='Temp (°C)')

plt.suptitle(f"Eagle at {frame.ground_truth_3d.round(1)} m", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Run Full Tracking Pipeline

In [ ]:
import time
from thermal_tracker.clustering import ClusteringConfig
from thermal_tracker.fusion import FusionConfig
from thermal_tracker.tracking import TrackingConfig
from thermal_tracker.voxel_grid import VoxelGridConfig
from thermal_tracker.pipeline import TrackingPipeline, generate_report
from thermal_tracker.tracking import TrackState

# Re-create engine (resets state)
engine = SimulationEngine(config)

# ROI sizing guide (voxel count = main speed factor):
#   800x800x150 @ 5m  = 768,000 voxels  → ~minutes/frame (TOO SLOW)
#   200x200x80  @ 10m =   3,200 voxels  → ~0.1s/frame    (GOOD)
#   400x400x100 @ 10m =  16,000 voxels  → ~0.5s/frame    (OK for enterprise Colab)
#
# The Lissajous trajectory with amplitudes [300,300,30] spans roughly
# X: [-300,300], Y: [-300,300], Z: [70,130]. We cover the first portion.

VOXEL_SIZE = 10.0  # meters — 10m gives good speed, 5m gives better accuracy
ROI_MIN = [-350.0, -350.0, 60.0]
ROI_MAX = [350.0, 350.0, 140.0]

nx = int((ROI_MAX[0] - ROI_MIN[0]) / VOXEL_SIZE)
ny = int((ROI_MAX[1] - ROI_MIN[1]) / VOXEL_SIZE)
nz = int((ROI_MAX[2] - ROI_MIN[2]) / VOXEL_SIZE)
total_voxels = nx * ny * nz
print(f"ROI: {nx}x{ny}x{nz} = {total_voxels:,} voxels (at {VOXEL_SIZE}m)")

pipeline = TrackingPipeline(
    cameras=engine.cameras,
    voxel_config=VoxelGridConfig(
        voxel_size=VOXEL_SIZE,
        roi_min=ROI_MIN,
        roi_max=ROI_MAX,
        occupancy_threshold=0.7,
        temporal_decay_rate=0.05,
    ),
    fusion_config=FusionConfig(
        detection_threshold_sigma=2.0,
        min_cameras_for_update=2,
    ),
    clustering_config=ClusteringConfig(
        method="connected_components",
        min_cluster_size=1,
        max_cluster_size=100,
    ),
    tracking_config=TrackingConfig(
        max_association_distance=30.0,
        min_hits_to_confirm=2,
        dt=config.run.dt,
    ),
)

# Run with progress output
print(f"\nRunning {config.run.num_frames} frames...")
tracking_results = []
t0 = time.time()
for i, frame_bundle in enumerate(engine.run_streaming()):
    result_tr = pipeline.process_frame(frame_bundle)
    tracking_results.append(result_tr)
    if (i + 1) % 10 == 0 or i == 0:
        elapsed = time.time() - t0
        det = "YES" if result_tr.estimated_position is not None else "no"
        n_vox = result_tr.num_active_voxels
        print(f"  Frame {i+1:3d}/{config.run.num_frames} | "
              f"{elapsed:.1f}s elapsed | "
              f"{result_tr.processing_time_ms:.0f}ms/frame | "
              f"active voxels: {n_vox:,} | "
              f"detection: {det}")

total = time.time() - t0
print(f"\nDone! {len(tracking_results)} frames in {total:.1f}s "
      f"({total/len(tracking_results)*1000:.0f}ms avg/frame)")

## 7. Evaluate Tracking Performance

In [ ]:
# Get ground truth from a fresh run (same seed = same trajectory)
engine2 = SimulationEngine(config)
sim_result = engine2.run(mode=SimulationOutputMode.BATCH)
gt = sim_result.trajectory

report = generate_report(tracking_results, gt, distance_threshold=20.0)

print("="*40)
print("   EVALUATION REPORT")
print("="*40)
print(f"Frames:              {report.num_frames}")
print(f"Detection rate:      {report.detection_rate:.1%}")
print(f"Mean position error: {report.mean_error:.2f} m")
print(f"Median error:        {report.median_error:.2f} m")
print(f"RMSE:                {report.rmse:.2f} m")
print(f"95th percentile:     {report.percentile_95:.2f} m")
print(f"Max error:           {report.max_error:.2f} m")
print(f"Precision:           {report.precision:.1%}")
print(f"Recall:              {report.recall:.1%}")
print(f"F1 score:            {report.f1_score:.1%}")
print(f"First detection:     frame {report.frames_to_first_detection}")
print(f"Longest gap:         {report.longest_detection_gap} frames")
print(f"FP rate:             {report.false_positive_rate:.1%}")
print(f"Avg time/frame:      {report.mean_processing_time_ms:.1f} ms")
print(f"Max time/frame:      {report.max_processing_time_ms:.1f} ms")

## 8. Plot Tracking Error Over Time

In [ ]:
from thermal_tracker.pipeline import compute_position_error

errors = compute_position_error(tracking_results, gt)
timestamps = gt[:, 0]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Position error
ax1.plot(timestamps, errors, 'b-', linewidth=1)
ax1.axhline(y=20.0, color='r', linestyle='--', alpha=0.5, label='Threshold (20m)')
ax1.set_ylabel('Position Error (m)')
ax1.set_title('Tracking Error Over Time')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Processing time
times = [r.processing_time_ms for r in tracking_results]
ax2.bar(timestamps, times, width=config.run.dt * 0.8, color='steelblue', alpha=0.7)
ax2.set_ylabel('Processing Time (ms)')
ax2.set_xlabel('Time (s)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Compare Ground Truth vs Tracked Trajectory

In [ ]:
# Extract tracked positions
tracked_frames = []
tracked_positions = []
for r in tracking_results:
    if r.estimated_position is not None:
        tracked_frames.append(r.frame_index)
        tracked_positions.append(r.estimated_position)

if tracked_positions:
    tracked = np.array(tracked_positions)

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')

    ax.plot(gt[:, 1], gt[:, 2], gt[:, 3], 'b-', linewidth=1.5, alpha=0.6, label='Ground truth')
    ax.plot(tracked[:, 0], tracked[:, 1], tracked[:, 2], 'r--', linewidth=1.5, label='Tracked')

    for cam_cfg in config.cameras:
        pos = cam_cfg.position
        ax.scatter(pos[0], pos[1], pos[2], c='orange', s=100, marker='^')

    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')
    ax.set_title('Ground Truth vs Tracked Trajectory')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No tracked positions available. Try adjusting ROI or fusion parameters.")

## 10. Experiment: Try Different Parameters

Modify the cells below to experiment with different settings.

In [ ]:
# Try random walk motion instead of Lissajous
config2 = load_config("configs/default_config.yaml")
config2.eagle.motion_type = "random_walk"
config2.run.num_frames = 50

engine3 = SimulationEngine(config2)
result3 = engine3.run(mode=SimulationOutputMode.BATCH)

traj3 = result3.trajectory
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(traj3[:, 1], traj3[:, 2], traj3[:, 3], 'r-', linewidth=1.5)
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
ax.set_title('Random Walk Trajectory')
plt.tight_layout()
plt.show()

In [ ]:
# Try with noise enabled
config3 = load_config("configs/default_config.yaml")
config3.rendering.noise.enabled = True
config3.rendering.noise.gaussian_std = 0.1
config3.run.num_frames = 10

engine4 = SimulationEngine(config3)
result4 = engine4.run(mode=SimulationOutputMode.BATCH)

frame = result4.frame_bundles[5]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, (cam_id, img) in enumerate(frame.camera_images.items()):
    im = axes[i].imshow(img, cmap='inferno', aspect='auto')
    axes[i].set_title(f"{cam_id} (with noise)")
    plt.colorbar(im, ax=axes[i], label='Temp (°C)')
plt.suptitle('Thermal Images with Sensor Noise', fontsize=12)
plt.tight_layout()
plt.show()